In [7]:
import json
from pathlib import Path


manual_file = Path("../data/manual_annotation_50.json")
pred_file = Path("../artworks/PROMPT_10f_20260603/clean_artworks_label_prediction.json")
output_file = Path("../artworks/PROMPT_10f_20260603/manual_labeled_examples_with_predictions.json")

marker = "/artworks/src/"


with manual_file.open("r", encoding="utf-8") as f:
    manual_data = json.load(f)

with pred_file.open("r", encoding="utf-8") as f:
    pred_data = json.load(f)


pred_by_path = {}

for pred in pred_data:
    pred_path = str(pred["file_path"])

    if marker in pred_path:
        pred_path = pred_path.split(marker, 1)[-1].lstrip("/")
    else:
        pred_path = pred_path.lstrip("/")

    pred_by_path[pred_path] = pred.get("predicted_labels", {
        "entities": [],
        "interaction": [],
        "outcome": []
    })


output_data = []

for item in manual_data:
    if "classification" not in item:
        continue

    item_path = str(item["file_path"])

    if marker in item_path:
        item_path = item_path.split(marker, 1)[-1].lstrip("/")
    else:
        item_path = item_path.lstrip("/")

    new_item = item.copy()
    new_item["file_path"] = item_path
    new_item["predicted_labels"] = pred_by_path.get(item_path, {
        "entities": [],
        "interaction": [],
        "outcome": []
    })

    output_data.append(new_item)


output_file.parent.mkdir(parents=True, exist_ok=True)

with output_file.open("w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)


print(f"Saved: {output_file}")
print(f"Exported examples: {len(output_data)}")

missing_predictions = sum(
    1
    for item in output_data
    if item["predicted_labels"] == {
        "entities": [],
        "interaction": [],
        "outcome": []
    }
)

print(f"Examples without matching prediction: {missing_predictions}")

Saved: ../artworks/PROMPT_10f_20260603/manual_labeled_examples_with_predictions.json
Exported examples: 51
Examples without matching prediction: 2
